# TECH2 Term Paper: Expectations about inflation, house prices, and the stock market

This notebook analyses household expectations in the United States using data from the Federal Reserve Bank of New York's [Survey of Consumer Expectations (SCE)](https://www.newyorkfed.org/microeconomics/sce#/).

## Setup

In [1]:
# Standard library
import glob
import os

# Third-party packages
import pandas as pd

# Folder containing all input data files
DATA_DIR = 'data'

***
## Part 1 — Reading in the data

### 1.1 Reading and combining the SCE files

Each survey wave is stored in its own file named `SCE-MMM-YYYY.csv`. We use `glob` to find all files matching this pattern (which excludes `CPI.csv`, the file used in Part 5), read each one with pandas and stack them into a single `DataFrame`. The files are semicolon-separated, and we parse the `date` column as dates.

In [2]:
def load_sce_data(data_dir=DATA_DIR, pattern='SCE-*.csv'):
    """
    Read all SCE survey files and combine them into a single DataFrame.

    Parameters
    ----------
    data_dir : str
        Folder containing the SCE CSV files.
    pattern : str
        Glob pattern matching the SCE file names.

    Returns
    -------
    pandas.DataFrame
        All survey waves stacked, sorted by respondent and interview date.
    """
    # Find all matching files; sort so the result is deterministic
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f'No files matching {pattern} in {data_dir}')

    # Read each file (semicolon-separated) and parse interview dates
    frames = [pd.read_csv(f, sep=';', parse_dates=['date']) for f in files]

    # Stack all waves into one DataFrame with a fresh index
    df = pd.concat(frames, ignore_index=True)

    # Sort by respondent and date so that each individual's observations
    # are in chronological order (needed for forward-filling in Part 2)
    df = df.sort_values(['userid', 'date'], ignore_index=True)

    return df

In [3]:
# Load all SCE survey waves
sce = load_sce_data()

# Show the first few rows to check the data were read correctly
sce.head()

,userid,wid,date,weight,female,educ,age,hispanic,black,couple,...,num_lit_q3,num_lit_q3_correct,num_lit_q5,num_lit_q5_correct,num_lit_q6,num_lit_q6_correct,num_lit_q8,num_lit_q8_correct,num_lit_q9,num_lit_q9_correct
0,70000220,201306,2013-06-04,16.3,1.0,3.0,28.0,0.0,1.0,0.0,...,100.0,0.0,100.0,1.0,5.0,1.0,NaN,NaN,NaN,NaN
1,70000224,201306,2013-06-03,0.2,0.0,4.0,65.0,0.0,0.0,1.0,...,10.0,1.0,100.0,1.0,5.0,1.0,NaN,NaN,NaN,NaN
2,70000234,201306,2013-06-17,4.1,1.0,3.0,41.0,0.0,0.0,1.0,...,10.0,1.0,100.0,1.0,5.0,1.0,NaN,NaN,NaN,NaN
3,70000238,201306,2013-06-13,3.0,0.0,3.0,74.0,0.0,0.0,1.0,...,10.0,1.0,1.0,0.0,5.0,1.0,NaN,NaN,NaN,NaN
4,70000238,201307,2013-07-10,1.9,0.0,3.0,74.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.2 Sample statistics

We report the number of unique individuals, observations, and survey waves, and the first and last interview dates. The same statistics are needed again in Part 2, so we write a reusable function.

In [4]:
def sample_statistics(df):
    """
    Compute basic sample-size statistics for SCE data.

    Parameters
    ----------
    df : pandas.DataFrame
        SCE data containing the columns `userid`, `wid` and `date`.

    Returns
    -------
    pandas.Series
        Sample statistics formatted for display (counts use a comma as
        thousands separator), indexed by a description of each statistic.
    """
    stats = {
        # Counts are formatted with a comma as thousands separator
        'Unique individuals': f'{df["userid"].nunique():,}',
        'Observations (rows)': f'{len(df):,}',
        'Unique survey waves': f'{df["wid"].nunique():,}',
        'First date': df['date'].min().date(),
        'Last date': df['date'].max().date(),
    }
    return pd.Series(stats, name='Value')

In [5]:
# Report sample statistics for the raw (unprocessed) data
sample_statistics(sce).to_frame()

,Value
Unique individuals,"23,369"
Observations (rows),"176,101"
Unique survey waves,139
First date,2013-06-01
Last date,2024-12-31
